In [13]:
# RAG Mini Project – Policy QA Assistant (NovaCart)
# LLM: Groq (LLaMA3)
# Retrieval: TF-IDF

import os
import re
import numpy as np
from groq import Groq
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Introduction
print("RAG Mini Project – Policy QA Assistant (NovaCart)\n")

# 2. Data Loading
DATA_DIR = "policy_docs" 

documents = []
doc_names = []

for file in os.listdir(DATA_DIR):
    if file.endswith(".txt"):
        with open(os.path.join(DATA_DIR, file), "r", encoding="utf-8") as f:
            documents.append(f.read())
            doc_names.append(file)

print(f"Loaded {len(documents)} policy documents")

# 3. Text Cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

documents = [clean_text(doc) for doc in documents]

# 4. Chunking Strategy
"""
Reasoning:
- 300-word chunks preserve policy semantics
- 50-word overlap prevents context loss
"""

def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))
        i += chunk_size - overlap
    return chunks

chunks = []
for doc in documents:
    chunks.extend(chunk_text(doc))

print(f"Created {len(chunks)} text chunks")

# 5. Embeddings (TF-IDF)
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(chunks)

# 6. Vector Store Setup
def retrieve(query, top_k=2):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, X)[0]
    top_indices = np.argsort(scores)[-top_k:][::-1]
    return [chunks[i] for i in top_indices]

# 7. Retrieval Demo
q_demo = "How long does a refund take?"
retrieved_chunks = retrieve(q_demo)

print("\nRetrieval Demo:")
print("Q:", q_demo)
print("-" * 75)
for c in retrieved_chunks:
    print(c[:300], "...\n")

# Groq Client Setup
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# 8. Prompt V1 (Basic)
PROMPT_V1 = """
Answer the question using the context below.

Context:
{context}

Question:
{question}

Answer:
"""

# 9. Prompt V2 (Improved – Hallucination Control)
PROMPT_V2 = """
You are a policy assistant for NovaCart.

Rules:
- Answer ONLY using the provided context.
- If the answer is not present, say "Information not available in the provided policy documents."
- Be concise and factual.

Context:
{context}

Question:
{question}

Answer:
"""

# RAG Ask Function
def ask(question, prompt):
    retrieved = retrieve(question)
    context = "\n\n".join(retrieved)
    final_prompt = prompt.format(context=context, question=question)

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",   # ✅ SUPPORTED MODEL
        messages=[{"role": "user", "content": final_prompt}],
        temperature=0
    )
    return response.choices[0].message.content

# 10. Prompt Comparison
q_compare = "Can I cancel my order after it is shipped?"

print("\nPrompt V1 Answer:")
print(ask(q_compare, PROMPT_V1))

print("\nPrompt V2 Answer:")
print(ask(q_compare, PROMPT_V2))

# 11. Evaluation Set + Results
eval_questions = [
    "How long does a refund take?",
    "Is cash on delivery available?",
    "What is the warranty period?",
    "Can I cancel after shipping?"
]

print("\nEvaluation Results:")
print("-" * 75)
print("{:<40} {:<30}".format("Question", "Answer Summary"))

for q in eval_questions:
    ans = ask(q, PROMPT_V2)
    print("{:<40} {:<30}".format(q, ans[:28] + "..."))

# 12. Edge Case Tests
edge_q = "Do you offer international drone delivery?"
print("\nEdge Case Test:")
print("Q:", edge_q)
print("A:", ask(edge_q, PROMPT_V2))



print("\n✅ RAG Pipeline Executed Successfully")


RAG Mini Project – Policy QA Assistant (NovaCart)

Loaded 4 policy documents
Created 4 text chunks

Retrieval Demo:
Q: How long does a refund take?
---------------------------------------------------------------------------
novacart refund policy at novacart, customer satisfaction is important to us. if you are not completely satisfied with your purchase, you may be eligible for a refund under the conditions below. refund eligibility - items must be returned within 30 days of delivery. - products must be unused, in ori ...

novacart cancellation policy we understand that plans change. you may cancel your order under the following conditions. when you can cancel orders can be cancelled: - within 12 hours of placing the order - before the order status changes to “shipped” when cancellation is not possible orders cannot b ...


Prompt V1 Answer:
No, you cannot cancel your order if it has already been shipped. According to Novacart's cancellation policy, orders cannot be cancelled if the o